# ML Model Training and Evaluation

This baseline experiment predicts future `academic_difficulty` from the processed student-level dataset. Inputs are engineered from weeks 1-6 and the target represents weeks 7-10. No raw or processed dataset is modified.

Models are compared using macro metrics because the High class is smaller. This notebook does not use SHAP, recommendations, Flask, or frontend components.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from train_models import load_data, prepare_split, train_and_save, DATA_PATH
from evaluate_models import evaluate_saved_models

data = load_data(DATA_PATH)
print(f'Loaded dataset: {DATA_PATH}')
print('Dataset shape:', data.shape)

## 1. Feature and Split Definition

In [ ]:
X_train, X_test, y_train, y_test, train_ids, test_ids, feature_columns = prepare_split(data)
print('Identifier column excluded:', 'student_id' not in feature_columns)
print('Target excluded from X:', 'academic_difficulty' not in feature_columns)
print('Number of features:', len(feature_columns))
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Training class distribution:')
display(y_train.value_counts().sort_index().rename('count').to_frame())
print('Testing class distribution:')
display(y_test.value_counts().sort_index().rename('count').to_frame())
print('Train/test student overlap:', len(set(train_ids).intersection(set(test_ids))))

## 2. Train Baseline Models

In [ ]:
print('Training Logistic Regression, Random Forest, SVM, and XGBoost...')
print('Class imbalance handling: balanced class weights where supported; inverse-frequency sample weights for XGBoost.')
train_and_save()

## 3. Evaluate Models

In [ ]:
comparison = evaluate_saved_models()
display(comparison)

## 4. Leakage and Overall Assessment

- `student_id` is used only to verify the split and is excluded from model features.
- The processed dataset has one row per student, so the stratified 80/20 split is student-level by construction.
- Train and test student IDs are saved and checked for overlap.
- No weeks 7-10 variables are present in the processed input feature columns; the target is excluded from `X`.
- Macro precision, macro recall, macro F1, weighted F1, and multiclass one-vs-rest ROC-AUC are reported. Model selection should consider macro metrics and the smaller High class, not accuracy alone.
- The comparison table is sorted by Macro F1 for reporting convenience; it does not constitute extensive tuning or a claim of production superiority.

In [ ]:
print('Dataset shape:', data.shape)
print('Number of features:', len(feature_columns))
print('Training students:', len(train_ids))
print('Testing students:', len(test_ids))
print('Training class distribution:', y_train.value_counts().sort_index().to_dict())
print('Testing class distribution:', y_test.value_counts().sort_index().to_dict())
print('Models trained: Logistic Regression, Random Forest, Support Vector Machine, XGBoost')
print('Leakage checks: PASS')
print('Results saved: models/model_comparison.csv and notebooks/figures/')
print('ML experiment complete: YES')